In [1]:
import pandas as pd
import numpy as np

URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

mathur = pd.read_csv(URL)

mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)

mathur = mathur.dropna(
    subset=["label", "Pattern String"]
).copy()

mathur["text"] = mathur["Pattern String"].str.strip()

mathur = mathur[
    mathur["text"] != ""
].drop_duplicates("text")

mathur["site"] = (
    mathur["Website Page"]
    .astype(str)
    .str.extract(r"https?://([^/]+)")[0]
    .str.replace(r"^www\.", "", regex=True)
)

mathur = (
    mathur
    .dropna(subset=["site"])[["text", "label", "site"]]
    .reset_index(drop=True)
)

print(
    len(mathur),
    "sentences from",
    mathur["site"].nunique(),
    "websites"
)

print(mathur["label"].value_counts())

1081 sentences from 792 websites
label
scarcity              418
social_proof          312
urgency               210
confirmshaming        129
hidden_information     12
Name: count, dtype: int64


In [2]:
shots = pd.read_csv("contextdp_ocr.csv")

print(len(shots), "screenshots")
print(shots["label"].value_counts())

shots.head(3)

440 screenshots
label
none                  237
preselection           98
urgency                43
hidden_information     34
scarcity               22
social_proof            6
Name: count, dtype: int64


,image,platform,their_labels,label,text
0,droom.in.png,web,ACTIVITY MESSAGE,social_proof,dr ©O iis Host Tasted 8 © location Vv (a) we #...
1,kiiroo.com.png,web,ACTIVITY MESSAGE,social_proof,Titan & Cliona Couple Set $229 or 4 interest-f...
2,nutrabay.com.png,web,ACTIVITY MESSAGE,social_proof,NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss W...


In [3]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),

    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),

    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),

    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),

    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),

    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),

    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),

    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),

    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),

    ("Everyone in dallas is buying the new ford bronco, now is your time",
     "social_proof"),

    ("The buy one get one free deal will end in 3 hours, shop now",
     "urgency"),

    ("the new harry potter movie is dropping on 22 september this year",
     "none"),

    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),

    ("We use cookies. Accept all or manage your preferences.",
     "none"),

    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(
    len(held_out),
    "held out sentences,",
    (held_out["label"] == "none").sum(),
    "of them none"
)

47 held out sentences, 23 of them none


In [4]:
from sklearn.model_selection import GroupShuffleSplit

# -----------------------------
# 1. Split Mathur sentences
# -----------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(mathur, groups=mathur["site"])
)

sentence_train = mathur.iloc[train_idx].reset_index(drop=True)
sentence_test = mathur.iloc[test_idx].reset_index(drop=True)


# -----------------------------
# 2. Load NEW screenshot split
# -----------------------------

screens_split = pd.read_csv("screens_split.csv")

print("screens_split columns:")
print(screens_split.columns.tolist())

print("\nSplit counts:")
print(screens_split["split"].value_counts())


# -----------------------------
# 3. Merge split into screenshots
# -----------------------------

shots = shots.merge(
    screens_split[["image", "split"]],
    on="image",
    how="left"
)

# Check that every screenshot got a split
print("\nMissing split values:")
print(shots["split"].isna().sum())


# -----------------------------
# 4. Create screenshot train/test
# -----------------------------

shot_train = shots[
    shots["split"].str.lower() == "train"
].reset_index(drop=True)

shot_test = shots[
    shots["split"].str.lower() == "test"
].reset_index(drop=True)


# -----------------------------
# 5. Combine training data
# -----------------------------

train = pd.concat(
    [
        sentence_train[["text", "label"]],
        shot_train[["text", "label"]]
    ],
    ignore_index=True
)


# -----------------------------
# 6. Print everything
# -----------------------------

print("\nSentence training rows:", len(sentence_train))
print("Sentence test rows:", len(sentence_test))

print("Screenshot training rows:", len(shot_train))
print("Screenshot test rows:", len(shot_test))

print("Combined training rows:", len(train))

print("\nTraining labels:")
print(train["label"].value_counts())

screens_split columns:
['image', 'split']

Split counts:
split
train    266
test     174
Name: count, dtype: int64

Missing split values:
0

Sentence training rows: 863
Sentence test rows: 218
Screenshot training rows: 266
Screenshot test rows: 174
Combined training rows: 1129

Training labels:
label
scarcity              351
social_proof          247
urgency               199
none                  139
confirmshaming        101
preselection           66
hidden_information     26
Name: count, dtype: int64


In [5]:
%pip -q install sentence-transformers

import torch
import sklearn
import sentence_transformers

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression


# Make sure Colab is using the T4 GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Go to Runtime > Change runtime type > T4 GPU."
    )


# Keep results reproducible
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


# Same encoder as last week
encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)


# Every piece of text must be encoded the same way
def encode_texts(texts):
    return encoder.encode(
        texts.fillna("").astype(str).tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )


# Convert training text into embeddings
X_train = encode_texts(train["text"])


# Same logistic regression settings as last week
model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)


# Train the model
model.fit(X_train, train["label"])


# Show information so we know everything worked
print("GPU:", torch.cuda.get_device_name(0))
print("Encoder token limit:", encoder.max_seq_length)
print("Training embedding shape:", X_train.shape)
print("Classes:", model.classes_)
print("scikit-learn version:", sklearn.__version__)
print("sentence-transformers version:", sentence_transformers.__version__)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

GPU: Tesla T4
Encoder token limit: 256
Training embedding shape: (1129, 384)
Classes: ['confirmshaming' 'hidden_information' 'none' 'preselection' 'scarcity'
 'social_proof' 'urgency']
scikit-learn version: 1.6.1
sentence-transformers version: 5.7.0


In [6]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

FOUR = ["confirmshaming", "scarcity", "social_proof", "urgency"]

def score_all(model_name, y_s, p_s, y_sh, p_sh, y_h, p_h):
    y_s, p_s, y_sh, p_sh, y_h, p_h = map(
        np.asarray,
        (y_s, p_s, y_sh, p_sh, y_h, p_h)
    )

    print(classification_report(
        y_sh,
        p_sh,
        zero_division=0
    ))

    rows = {
        "sent_f1_four": f1_score(
            y_s,
            p_s,
            labels=FOUR,
            average="macro",
            zero_division=0
        ),

        "shot_accuracy": accuracy_score(
            y_sh,
            p_sh
        ),

        "shot_always_none": (
            y_sh == "none"
        ).mean(),

        "shot_dark_f1": f1_score(
            y_sh != "none",
            p_sh != "none"
        ),

        "heldout_none_passed": int(
            ((y_h == "none") & (p_h == "none")).sum()
        ),

        "heldout_dark_right": int(
            ((y_h != "none") & (p_h == y_h)).sum()
        )
    }

    return pd.DataFrame([
        {
            "model": model_name,
            "metric": k,
            "value": v
        }
        for k, v in rows.items()
    ])

In [7]:
# Convert the test text into embeddings

X_sentence_test = encode_texts(sentence_test["text"])
X_shot_test = encode_texts(shot_test["text"])
X_held_out = encode_texts(held_out["text"])


# Get the correct answers

y_s = sentence_test["label"].to_numpy()
y_sh = shot_test["label"].to_numpy()
y_h = held_out["label"].to_numpy()


# Ask our trained model for its predictions

p_s = model.predict(X_sentence_test)
p_sh = model.predict(X_shot_test)
p_h = model.predict(X_held_out)


# Score the model

results_whole = score_all(
    "embeddings_whole",
    y_s,
    p_s,
    y_sh,
    p_sh,
    y_h,
    p_h
)


print("\nRESULTS:")
print(results_whole)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

                    precision    recall  f1-score   support

hidden_information       0.23      0.26      0.24        19
              none       0.73      0.77      0.75        98
      preselection       0.51      0.62      0.56        32
          scarcity       0.00      0.00      0.00         5
      social_proof       0.00      0.00      0.00         3
           urgency       0.50      0.24      0.32        17

          accuracy                           0.60       174
         macro avg       0.33      0.31      0.31       174
      weighted avg       0.58      0.60      0.58       174


RESULTS:
              model               metric      value
0  embeddings_whole         sent_f1_four   0.978920
1  embeddings_whole        shot_accuracy   0.597701
2  embeddings_whole     shot_always_none   0.563218
3  embeddings_whole         shot_dark_f1   0.653061
4  embeddings_whole  heldout_none_passed   2.000000
5  embeddings_whole   heldout_dark_right  21.000000


In [8]:
# Make the screenshot prediction table

preds_whole = shot_test[["image", "label"]].copy()

preds_whole["predicted"] = p_sh


# Save screenshot predictions

preds_whole.to_csv(
    "preds_Prasanna_v2.csv",
    index=False
)


# Save the score table

results_whole.to_csv(
    "results_Prasanna_v2.csv",
    index=False
)


print("Saved:")
print("preds_Prasanna_v2.csv")
print("results_Prasanna_v2.csv")

Saved:
preds_Prasanna_v2.csv
results_Prasanna_v2.csv


In [9]:
# Window settings
WINDOW_WORDS = 15
WINDOW_STEP = 8
CUTOFF = 0.50


# Break long OCR text into overlapping pieces
def chunk_text(text):
    words = str(text).split()

    # If text is already short, just use the whole thing
    if len(words) <= WINDOW_WORDS:
        return [" ".join(words)]

    chunks = []

    for start in range(0, len(words), WINDOW_STEP):
        chunk = words[start:start + WINDOW_WORDS]

        if len(chunk) == 0:
            break

        chunks.append(" ".join(chunk))

        # Stop once we reach the end
        if start + WINDOW_WORDS >= len(words):
            break

    return chunks


# Quick example so we can see how it works
example = """
Shop our newest collection today. Free shipping on all orders.
Only 2 items left in stock. Customers love this product.
Returns accepted within 30 days.
"""

print(chunk_text(example))

['Shop our newest collection today. Free shipping on all orders. Only 2 items left in', 'all orders. Only 2 items left in stock. Customers love this product. Returns accepted within', 'Customers love this product. Returns accepted within 30 days.']


In [10]:
def predict_frame_windows(frame, clf):
    all_chunks = []
    owners = []

    # Make chunks for every screenshot
    for row_num, text in enumerate(frame["text"].fillna("").astype(str)):
        chunks = chunk_text(text)

        all_chunks.extend(chunks)
        owners.extend([row_num] * len(chunks))

    # Convert all chunks into embeddings
    X_chunks = encode_texts(pd.Series(all_chunks))

    # Get probabilities for every class
    probabilities = clf.predict_proba(X_chunks)

    classes = np.asarray(clf.classes_)

    # We only want dark-pattern categories here, not "none"
    dark_mask = classes != "none"

    dark_classes = classes[dark_mask]
    dark_probabilities = probabilities[:, dark_mask]

    owners = np.asarray(owners)

    predictions = []

    # Go screenshot by screenshot
    for row_num in range(len(frame)):

        # Grab all chunk probabilities belonging to this screenshot
        row_probs = dark_probabilities[
            owners == row_num
        ]

        # Find the strongest dark-pattern probability
        flat_index = np.argmax(row_probs)

        chunk_index, class_index = np.unravel_index(
            flat_index,
            row_probs.shape
        )

        strongest_probability = row_probs[
            chunk_index,
            class_index
        ]

        strongest_category = dark_classes[
            class_index
        ]

        # Our cutoff is 0.50
        if strongest_probability > CUTOFF:
            predictions.append(
                strongest_category
            )
        else:
            predictions.append(
                "none"
            )

    return np.asarray(predictions)

In [11]:
# Predict the screenshot TEST set using windows

p_sh_windows = predict_frame_windows(
    shot_test,
    model
)

print("Number of predictions:", len(p_sh_windows))
print("Number of screenshot test rows:", len(shot_test))

print("\nFirst 10 predictions:")
print(p_sh_windows[:10])

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Number of predictions: 174
Number of screenshot test rows: 174

First 10 predictions:
['urgency' 'urgency' 'urgency' 'urgency' 'none' 'none' 'urgency'
 'hidden_information' 'urgency' 'none']


In [12]:
results_windows = score_all(
    "embeddings_windows",
    y_s,
    p_s,
    y_sh,
    p_sh_windows,
    y_h,
    p_h
)

print("\nWINDOW RESULTS:")
print(results_windows)

                    precision    recall  f1-score   support

hidden_information       0.33      0.21      0.26        19
              none       0.77      0.69      0.73        98
      preselection       0.49      0.69      0.57        32
          scarcity       0.50      0.20      0.29         5
      social_proof       0.00      0.00      0.00         3
           urgency       0.50      0.76      0.60        17

          accuracy                           0.62       174
         macro avg       0.43      0.43      0.41       174
      weighted avg       0.62      0.62      0.61       174


WINDOW RESULTS:
                model               metric      value
0  embeddings_windows         sent_f1_four   0.978920
1  embeddings_windows        shot_accuracy   0.620690
2  embeddings_windows     shot_always_none   0.563218
3  embeddings_windows         shot_dark_f1   0.691358
4  embeddings_windows  heldout_none_passed   2.000000
5  embeddings_windows   heldout_dark_right  21.000000


In [13]:
preds_windows = shot_test[
    ["image", "label"]
].copy()

preds_windows["predicted"] = p_sh_windows

preds_windows.to_csv(
    "preds_Prasanna_windows.csv",
    index=False
)

print("Saved preds_Prasanna_windows.csv")

Saved preds_Prasanna_windows.csv


In [14]:
results_all = pd.concat(
    [
        results_whole,
        results_windows
    ],
    ignore_index=True
)

results_all.to_csv(
    "results_Prasanna_v2.csv",
    index=False
)

print(results_all)

print("\nUpdated results_Prasanna_v2.csv")

                 model               metric      value
0     embeddings_whole         sent_f1_four   0.978920
1     embeddings_whole        shot_accuracy   0.597701
2     embeddings_whole     shot_always_none   0.563218
3     embeddings_whole         shot_dark_f1   0.653061
4     embeddings_whole  heldout_none_passed   2.000000
5     embeddings_whole   heldout_dark_right  21.000000
6   embeddings_windows         sent_f1_four   0.978920
7   embeddings_windows        shot_accuracy   0.620690
8   embeddings_windows     shot_always_none   0.563218
9   embeddings_windows         shot_dark_f1   0.691358
10  embeddings_windows  heldout_none_passed   2.000000
11  embeddings_windows   heldout_dark_right  21.000000

Updated results_Prasanna_v2.csv


In [15]:
# Get ONLY the none screenshots from the TRAIN split

none_train_shots = shot_train[
    shot_train["label"] == "none"
].copy()


# Turn those none screenshots into short chunks

none_chunk_rows = []

for text in none_train_shots["text"].fillna("").astype(str):

    chunks = chunk_text(text)

    for chunk in chunks:
        none_chunk_rows.append({
            "text": chunk,
            "label": "none"
        })


none_chunks = pd.DataFrame(
    none_chunk_rows
)


print("None training screenshots:", len(none_train_shots))
print("None chunks created:", len(none_chunks))

none_chunks.head()

None training screenshots: 139
None chunks created: 902


,text,label
0,4 100% VEGAN NOW AND FOREVER. LEARN MORE. 9 kn...,none
1,9 know your location Block | Allow \ For Profe...,none
2,For Professionals @ Stores &Salons Signin 6 6 ...,none
3,HAIRCARE STYLING SKINCARE BODYCARE MAKEUP MEN'...,none
4,HAIRQUIZ OURSTORY OFFERS -—_ SERVICES Q HAIR C...,none


In [16]:
# Add the short NONE chunks to our original training data

train_nonechunks = pd.concat(
    [
        train[["text", "label"]],
        none_chunks[["text", "label"]]
    ],
    ignore_index=True
)


print("Original training rows:", len(train))
print("New training rows:", len(train_nonechunks))

print("\nNew training labels:")
print(train_nonechunks["label"].value_counts())

Original training rows: 1129
New training rows: 2031

New training labels:
label
none                  1041
scarcity               351
social_proof           247
urgency                199
confirmshaming         101
preselection            66
hidden_information      26
Name: count, dtype: int64


In [17]:
# Encode the new training set

X_train_nonechunks = encode_texts(
    train_nonechunks["text"]
)


# Create a NEW logistic regression model
# with the SAME settings

model_nonechunks = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)


# Train Version 3

model_nonechunks.fit(
    X_train_nonechunks,
    train_nonechunks["label"]
)


print("Version 3 trained successfully.")
print("Training shape:", X_train_nonechunks.shape)
print("Classes:", model_nonechunks.classes_)

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Version 3 trained successfully.
Training shape: (2031, 384)
Classes: ['confirmshaming' 'hidden_information' 'none' 'preselection' 'scarcity'
 'social_proof' 'urgency']


In [18]:
# Sentence predictions with the new model

p_s_nonechunks = model_nonechunks.predict(
    X_sentence_test
)


# Held-out predictions with the new model

p_h_nonechunks = model_nonechunks.predict(
    X_held_out
)


# Screenshot predictions using WINDOWS
# with the new none-chunk model

p_sh_nonechunks = predict_frame_windows(
    shot_test,
    model_nonechunks
)


# Score Version 3

results_nonechunks = score_all(
    "embeddings_windows_nonechunks",
    y_s,
    p_s_nonechunks,
    y_sh,
    p_sh_nonechunks,
    y_h,
    p_h_nonechunks
)


print("\nWINDOWS + NONE CHUNKS RESULTS:")
print(results_nonechunks)

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

                    precision    recall  f1-score   support

hidden_information       0.29      0.21      0.24        19
              none       0.75      0.71      0.73        98
      preselection       0.49      0.62      0.55        32
          scarcity       0.50      0.20      0.29         5
      social_proof       0.00      0.00      0.00         3
           urgency       0.55      0.71      0.62        17

          accuracy                           0.61       174
         macro avg       0.43      0.41      0.40       174
      weighted avg       0.61      0.61      0.61       174


WINDOWS + NONE CHUNKS RESULTS:
                           model               metric      value
0  embeddings_windows_nonechunks         sent_f1_four   0.965930
1  embeddings_windows_nonechunks        shot_accuracy   0.614943
2  embeddings_windows_nonechunks     shot_always_none   0.563218
3  embeddings_windows_nonechunks         shot_dark_f1   0.675159
4  embeddings_windows_nonechunks  heldou

In [19]:
# Save Version 3 screenshot predictions

preds_nonechunks = shot_test[
    ["image", "label"]
].copy()

preds_nonechunks["predicted"] = p_sh_nonechunks

preds_nonechunks.to_csv(
    "preds_Prasanna_windows_nonechunks.csv",
    index=False
)

print("Saved preds_Prasanna_windows_nonechunks.csv")

Saved preds_Prasanna_windows_nonechunks.csv


In [20]:
# Put ALL THREE model versions into one results file

results_all = pd.concat(
    [
        results_whole,
        results_windows,
        results_nonechunks
    ],
    ignore_index=True
)

results_all.to_csv(
    "results_Prasanna_v2.csv",
    index=False
)

print(results_all)

print("\nSaved final results_Prasanna_v2.csv")

                            model               metric      value
0                embeddings_whole         sent_f1_four   0.978920
1                embeddings_whole        shot_accuracy   0.597701
2                embeddings_whole     shot_always_none   0.563218
3                embeddings_whole         shot_dark_f1   0.653061
4                embeddings_whole  heldout_none_passed   2.000000
5                embeddings_whole   heldout_dark_right  21.000000
6              embeddings_windows         sent_f1_four   0.978920
7              embeddings_windows        shot_accuracy   0.620690
8              embeddings_windows     shot_always_none   0.563218
9              embeddings_windows         shot_dark_f1   0.691358
10             embeddings_windows  heldout_none_passed   2.000000
11             embeddings_windows   heldout_dark_right  21.000000
12  embeddings_windows_nonechunks         sent_f1_four   0.965930
13  embeddings_windows_nonechunks        shot_accuracy   0.614943
14  embedd

In [21]:
# If text has 30 words or fewer, score the whole text.
# If it has more than 30 words, use windows.

LONG_TEXT_WORDS = 30


def get_dark_probabilities(text):

    words = str(text).split()

    # SHORT TEXT
    if len(words) <= LONG_TEXT_WORDS:
        pieces = [str(text)]

    # LONG TEXT
    else:
        pieces = chunk_text(text)


    # Convert text/chunks to embeddings
    X = encode_texts(
        pd.Series(pieces)
    )


    # Get probabilities from Version 3 model
    probabilities = model_nonechunks.predict_proba(X)

    classes = np.asarray(
        model_nonechunks.classes_
    )


    dark_probabilities = {}


    # Look at each dark-pattern category
    for class_index, category in enumerate(classes):

        # Skip none because categories should
        # only contain dark patterns
        if category == "none":
            continue

        # For long text:
        # take the strongest probability
        # across all windows
        strongest = probabilities[
            :,
            class_index
        ].max()

        dark_probabilities[
            category
        ] = float(strongest)


    return dark_probabilities

In [22]:
def predict(text):

    probabilities = get_dark_probabilities(text)


    # Keep every dark category above our 0.50 cutoff
    categories = {
        category: round(probability, 4)
        for category, probability in probabilities.items()
        if probability > CUTOFF
    }


    # Nothing passed the cutoff
    if len(categories) == 0:

        return {
            "is_dark": False,
            "categories": {},
            "top_category": "none"
        }


    # Find the category with the highest probability
    top_category = max(
        categories,
        key=categories.get
    )


    return {
        "is_dark": True,
        "categories": categories,
        "top_category": top_category
    }

In [23]:
predict(
    "Only 2 left in stock"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'is_dark': True,
 'categories': {'scarcity': 0.9837},
 'top_category': 'scarcity'}

In [24]:
predict(
    "Free shipping on orders over $50"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'is_dark': False, 'categories': {}, 'top_category': 'none'}

In [25]:
# Find the longest OCR text in the screenshot TEST set

word_counts = (
    shot_test["text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

long_index = word_counts.idxmax()

long_text = shot_test.loc[
    long_index,
    "text"
]


print(
    "Image:",
    shot_test.loc[long_index, "image"]
)

print(
    "True label:",
    shot_test.loc[long_index, "label"]
)

print(
    "Word count:",
    len(str(long_text).split())
)

print(
    "\nOCR preview:"
)

print(
    str(long_text)[:500]
)

print(
    "\nPrediction:"
)

print(
    predict(long_text)
)

Image: nutrabay.com.png
True label: social_proof
Word count: 221

OCR preview:
NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss WELLNESS PROTEIN POWDERS VITAMINS CRAZY DEALS TOP 10 LOGIN/REGISTER CART/70 a) jonmx% 7 Giocovare creme Ultimate Nutrition Ultimate Nutrition Prostar 100 Whey Protein Sek k kk 697 Reviews @ Sold & Fulfilled By Nutrabay.com - v Brand Authorized Free & Fast Delivery Shipped within 1 day. Free shipping on orders above 2350. (J Genuine Products All our products are far from expiry, and procured directly from the brand or authorized importers of the brand.

Prediction:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'is_dark': False, 'categories': {}, 'top_category': 'none'}


In [26]:
from pathlib import Path

def load_salesy_files(folder):

    folder = Path(folder)

    files = list(folder.glob("*.csv"))

    if len(files) == 0:
        raise ValueError("No CSV files found.")

    all_data = []

    for file in files:

        df = pd.read_csv(file)

        required = {"text", "label", "author"}

        if not required.issubset(df.columns):
            raise ValueError(
                f"{file.name} must contain text, label, author"
            )

        all_data.append(
            df[["text", "label", "author"]]
        )


    combined = pd.concat(
        all_data,
        ignore_index=True
    )

    return combined

In [27]:
# Create a fake folder

fake_folder = Path("fake_salesy")

fake_folder.mkdir(
    exist_ok=True
)


# Fake author 1

alex = pd.DataFrame([
    {
        "text": "Free shipping on orders over $50",
        "label": "none",
        "author": "Alex"
    },
    {
        "text": "New colors available this season",
        "label": "none",
        "author": "Alex"
    }
])


# Fake author 2

jordan = pd.DataFrame([
    {
        "text": "Student discount available with valid ID",
        "label": "none",
        "author": "Jordan"
    },
    {
        "text": "Returns accepted within 30 days",
        "label": "none",
        "author": "Jordan"
    }
])


alex.to_csv(
    fake_folder / "alex.csv",
    index=False
)

jordan.to_csv(
    fake_folder / "jordan.csv",
    index=False
)


print("Fake files created.")

Fake files created.


In [28]:
fake_salesy = load_salesy_files(
    "fake_salesy"
)

print(fake_salesy)

                                       text label  author
0  Student discount available with valid ID  none  Jordan
1           Returns accepted within 30 days  none  Jordan
2          Free shipping on orders over $50  none    Alex
3          New colors available this season  none    Alex


In [29]:
def evaluate_by_author(salesy_df):

    results = []

    # Our Step 4 training data
    base_X = X_train_nonechunks
    base_y = train_nonechunks["label"].to_numpy()


    # Test each author separately
    for author in sorted(salesy_df["author"].unique()):

        print("\nTesting author:", author)

        # This author's sentences = TEST
        author_test = salesy_df[
            salesy_df["author"] == author
        ].copy()


        # Everyone else's sentences = extra TRAINING data
        other_authors = salesy_df[
            salesy_df["author"] != author
        ].copy()


        # Encode the other authors' sentences
        if len(other_authors) > 0:

            X_other = encode_texts(
                other_authors["text"]
            )

            X_augmented = np.vstack([
                base_X,
                X_other
            ])

            y_augmented = np.concatenate([
                base_y,
                other_authors["label"].to_numpy()
            ])

        else:

            X_augmented = base_X
            y_augmented = base_y


        # Train a NEW model for this author
        author_model = LogisticRegression(
            C=1.0,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=2000,
            random_state=42
        )

        author_model.fit(
            X_augmented,
            y_augmented
        )


        # Test on this author's sentences
        X_author = encode_texts(
            author_test["text"]
        )

        author_predictions = author_model.predict(
            X_author
        )


        # Also test the 47 held-out sentences
        heldout_predictions = author_model.predict(
            X_held_out
        )


        # Count results
        author_none_count = int(
            (author_predictions == "none").sum()
        )

        heldout_none_passed = int(
            (
                (y_h == "none") &
                (heldout_predictions == "none")
            ).sum()
        )


        results.append({
            "author": author,
            "author_sentences": len(author_test),
            "came_out_none": author_none_count,
            "heldout_none_passed": heldout_none_passed
        })


    return pd.DataFrame(results)

In [30]:
fake_author_results = evaluate_by_author(
    fake_salesy
)

print("\nAUTHOR TEST RESULTS:")
print(fake_author_results)


Testing author: Alex


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Testing author: Jordan


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


AUTHOR TEST RESULTS:
   author  author_sentences  came_out_none  heldout_none_passed
0    Alex                 2              0                    9
1  Jordan                 2              0                    9


In [31]:
# Count how many TRUE none screenshots each version incorrectly flagged as dark

whole_none_flagged = int(
    ((y_sh == "none") & (p_sh != "none")).sum()
)

windows_none_flagged = int(
    ((y_sh == "none") & (p_sh_windows != "none")).sum()
)

nonechunks_none_flagged = int(
    ((y_sh == "none") & (p_sh_nonechunks != "none")).sum()
)


print("Whole-page none screenshots flagged:", whole_none_flagged)
print("Windows none screenshots flagged:", windows_none_flagged)
print("Windows + none chunks flagged:", nonechunks_none_flagged)

Whole-page none screenshots flagged: 23
Windows none screenshots flagged: 30
Windows + none chunks flagged: 28


In [32]:
notes = f"""The whole-page model had 0.598 screenshot accuracy and 0.653 dark F1, while window scoring improved to 0.621 accuracy and 0.691 dark F1. Windows performed best on screenshot dark F1, although windows plus none chunks improved held-out none recognition from 2 of 23 to 9 of 23. Whole-page scoring incorrectly flagged {whole_none_flagged} none screenshots, windows flagged {windows_none_flagged}, and windows plus none chunks flagged {nonechunks_none_flagged}. Windows can flag more none screenshots because short OCR fragments can look suspicious by themselves, while training on none chunks gives the model more examples of harmless short text."""

with open(
    "notes_Prasanna.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(notes)

print(notes)
print("\nSaved notes_Prasanna.md")

The whole-page model had 0.598 screenshot accuracy and 0.653 dark F1, while window scoring improved to 0.621 accuracy and 0.691 dark F1. Windows performed best on screenshot dark F1, although windows plus none chunks improved held-out none recognition from 2 of 23 to 9 of 23. Whole-page scoring incorrectly flagged 23 none screenshots, windows flagged 30, and windows plus none chunks flagged 28. Windows can flag more none screenshots because short OCR fragments can look suspicious by themselves, while training on none chunks gives the model more examples of harmless short text.

Saved notes_Prasanna.md
